# Data Cleaning & Processing

## Imports

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

## Load the Data

In [2]:
cwd = Path.cwd()
root = next(p for p in [cwd, *cwd.parents] if (p / "data" / "raw" / "survey_data.csv").exists())

survey_path = root / "data" / "raw" / "survey_data.csv"
codelist_path = root / "data" / "raw" / "codelist.csv"
df = pd.read_csv(survey_path)
codelist = pd.read_csv(codelist_path)
print(f"Loaded {survey_path}: {df.shape[0]} rows x {df.shape[1]} columns")

Loaded /Users/iansargent/Precision_Interview_Assessment/data/raw/survey_data.csv: 100 rows x 8 columns


## Standardize Column Names

In [3]:
df_clean = df.copy()
df_clean.columns = (df_clean.columns.str.strip().str.lower().str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))
required = {"respondent", "sex", "location", "q1", "q2", "q3", "q4", "q5"}
missing = required.difference(df_clean.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
print("Standardized columns:", df_clean.columns.tolist())

Standardized columns: ['respondent', 'sex', 'location', 'q1', 'q2', 'q3', 'q4', 'q5']


## Verify the 1-5 Encodings

The codebook defines 1 as the most positive/aware response and 5 as "Don't know," "Unsure," or "Not aware at all." These codes remain in their original codebook direction: no scale reversal is applied.

In [4]:
ordinal_columns = ["q1", "q2", "q3", "q5"]
for col in ordinal_columns:
    values = set(pd.to_numeric(df_clean[col], errors="raise").unique())
    unexpected = values.difference({1, 2, 3, 4, 5})
    if unexpected:
        raise ValueError(f"{col} contains codes outside 1-5: {sorted(unexpected)}")

q4_values = set(pd.to_numeric(df_clean["q4"], errors="raise").unique())
if not q4_values.issubset({1, 2, 3, 4, 5}):
    raise ValueError(f"q4 contains unexpected codes: {sorted(q4_values)}")

print("Encoding check passed: all responses use codes 1-5 in codebook order.")
print(df_clean[["q1", "q2", "q3", "q5"]].apply(lambda s: s.value_counts().sort_index().to_dict()).to_dict())

Encoding check passed: all responses use codes 1-5 in codebook order.
{'q1': {1: 19, 2: 24, 3: 25, 4: 20, 5: 12}, 'q2': {1: 26, 2: 25, 3: 20, 4: 19, 5: 10}, 'q3': {1: 49, 2: 24, 3: 19, 4: 7, 5: 1}, 'q5': {1: 16, 2: 19, 3: 23, 4: 18, 5: 24}}


## Missingness and Duplicates

In [5]:
missing_summary = pd.DataFrame({"missing_count": df_clean.isna().sum(), "missing_percentage": df_clean.isna().mean().mul(100).round(2), "dtype": df_clean.dtypes.astype(str)}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Duplicate rows:", df_clean.duplicated().sum())

,missing_count,missing_percentage,dtype
respondent,0,0.0,int64
sex,0,0.0,object
location,0,0.0,object
q1,0,0.0,int64
q2,0,0.0,int64
q3,0,0.0,int64
q4,0,0.0,int64
q5,0,0.0,int64


Duplicate rows: 0


## Recode Q4 as a Categorical Label

In [6]:
q4_row = codelist.loc[codelist.iloc[:, 0].astype(str).str.strip().str.upper() == "Q4"]
if len(q4_row) != 1:
    raise ValueError("Expected exactly one Q4 row in the codebook")
q4_row = q4_row.iloc[0]
q4_map = {int(code): str(q4_row[str(code)]).strip() for code in range(1, 6)}
df_clean["q4"] = pd.to_numeric(df_clean["q4"], errors="raise").map(q4_map)
display(df_clean["q4"].value_counts(dropna=False).rename("count").to_frame())

,count
q4,
Land & Wildlife,29
Climate Change,28
Clean Air & Water,24
Environmental Justice,16
Something Else / Don't Know,3


## Final Structure and Saved Output

In [7]:
display(df_clean.head())
print("Final shape:", df_clean.shape)
print("Final dtypes:")
print(df_clean.dtypes)
clean_path = root / "data" / "clean" / "clean_data.csv"
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)
print(f"Saved cleaned data to {clean_path}")

,respondent,sex,location,q1,q2,q3,q4,q5
0,1,M,MT,4,3,1,Climate Change,5
1,2,F,CO,2,1,1,Environmental Justice,2
2,3,M,CO,3,3,2,Clean Air & Water,4
3,4,M,MT,3,1,1,Clean Air & Water,2
4,5,M,MT,2,2,1,Land & Wildlife,1


Final shape: (100, 8)
Final dtypes:
respondent     int64
sex           object
location      object
q1             int64
q2             int64
q3             int64
q4            object
q5             int64
dtype: object
Saved cleaned data to /Users/iansargent/Precision_Interview_Assessment/data/clean/clean_data.csv
